In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 9.8 MB/s eta 0:00:00


In [2]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # trial object is passed when the study.optimize() fxn get triggered
    # Suggest values for the hyperparameters based on the trail object which was
      # tuned by the TPE sampler
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize
          # this is the value being returned to set inside the study object


In [30]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2025-07-14 10:37:40,712] A new study created in memory with name: no-name-99537fff-f249-4267-b4ae-bd41fbcc2859
[I 2025-07-14 10:37:41,590] Trial 0 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 161, 'max_depth': 20}. Best is trial 0 with value: 0.7709497206703911.
[I 2025-07-14 10:37:41,939] Trial 1 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 90, 'max_depth': 19}. Best is trial 1 with value: 0.7765363128491621.
[I 2025-07-14 10:37:42,548] Trial 2 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 159, 'max_depth': 14}. Best is trial 1 with value: 0.7765363128491621.
[I 2025-07-14 10:37:42,772] Trial 3 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 54, 'max_depth': 19}. Best is trial 1 with value: 0.7765363128491621.
[I 2025-07-14 10:37:43,382] Trial 4 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 163, 'max_depth': 10}. Best is trial 1 with value: 0.776536

In [31]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7839851024208566
Best hyperparameters: {'n_estimators': 73, 'max_depth': 18}


In [32]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.76


## Samplers in Optuna

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [16]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2025-07-14 10:32:55,611] A new study created in memory with name: no-name-e90d9d6f-3362-4629-bc3f-77cd7a043ad2
[I 2025-07-14 10:32:56,227] Trial 0 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 156, 'max_depth': 20}. Best is trial 0 with value: 0.7728119180633147.
[I 2025-07-14 10:32:56,475] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 65, 'max_depth': 9}. Best is trial 0 with value: 0.7728119180633147.
[I 2025-07-14 10:32:56,787] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 75, 'max_depth': 16}. Best is trial 0 with value: 0.7728119180633147.
[I 2025-07-14 10:32:57,092] Trial 3 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 78, 'max_depth': 11}. Best is trial 0 with value: 0.7728119180633147.
[I 2025-07-14 10:32:57,444] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 87, 'max_depth': 12}. Best is trial 0 with value: 0.772811918

In [19]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 57, 'max_depth': 7}


In [20]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


In [21]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [22]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2025-07-14 10:34:36,393] A new study created in memory with name: no-name-264b61c3-9347-4b35-8b3b-cdd5dc9ec408
[I 2025-07-14 10:34:36,757] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-07-14 10:34:37,333] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-07-14 10:34:37,535] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-07-14 10:34:37,930] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-07-14 10:34:38,316] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [23]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [24]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


## Optuna Visualizations

In [33]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [34]:
# 1. Optimization History
plot_optimization_history(study).show()

In [35]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [36]:
# 3. Slice Plot
plot_slice(study).show()

In [37]:
# 4. Contour Plot
plot_contour(study).show()

In [38]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

## Optimizing Multiple ML Models

In [39]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [40]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [41]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-07-14 10:58:43,743] A new study created in memory with name: no-name-7997a4d6-4651-48bd-bf6b-53d7684ae54e
[I 2025-07-14 10:58:43,972] Trial 0 finished with value: 0.7690875232774674 and parameters: {'classifier': 'RandomForest', 'n_estimators': 66, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 3, 'bootstrap': False}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-07-14 10:58:44,389] Trial 1 finished with value: 0.7746741154562384 and parameters: {'classifier': 'RandomForest', 'n_estimators': 139, 'max_depth': 18, 'min_samples_split': 8, 'min_samples_leaf': 8, 'bootstrap': False}. Best is trial 1 with value: 0.7746741154562384.
[I 2025-07-14 10:58:45,246] Trial 2 finished with value: 0.739292364990689 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 58, 'learning_rate': 0.020220688439419297, 'max_depth': 15, 'min_samples_split': 6, 'min_samples_leaf': 2}. Best is trial 1 with value: 0.7746741154562384.
[I 2025-07-14 10:58:45,267] Trial 3

In [42]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.12131663935392521, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [43]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.769088,2025-07-14 10:58:43.744388,2025-07-14 10:58:43.972334,0 days 00:00:00.227946,NaN,False,RandomForest,NaN,NaN,NaN,10.0,3.0,4.0,66.0,COMPLETE
1,1,0.774674,2025-07-14 10:58:43.973059,2025-07-14 10:58:44.389540,0 days 00:00:00.416481,NaN,False,RandomForest,NaN,NaN,NaN,18.0,8.0,8.0,139.0,COMPLETE
2,2,0.739292,2025-07-14 10:58:44.390339,2025-07-14 10:58:45.246732,0 days 00:00:00.856393,NaN,NaN,GradientBoosting,NaN,NaN,0.020221,15.0,2.0,6.0,58.0,COMPLETE
3,3,0.694600,2025-07-14 10:58:45.247484,2025-07-14 10:58:45.267879,0 days 00:00:00.020395,14.686503,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.756052,2025-07-14 10:58:45.268676,2025-07-14 10:58:46.897919,0 days 00:00:01.629243,NaN,True,RandomForest,NaN,NaN,NaN,19.0,6.0,10.0,264.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.743017,2025-07-14 10:59:07.271643,2025-07-14 10:59:07.299509,0 days 00:00:00.027866,0.117931,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.785847,2025-07-14 10:59:07.300428,2025-07-14 10:59:07.339029,0 days 00:00:00.038601,6.055523,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.772812,2025-07-14 10:59:07.339641,2025-07-14 10:59:07.374655,0 days 00:00:00.035014,0.152089,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.744879,2025-07-14 10:59:07.375276,2025-07-14 10:59:08.806915,0 days 00:00:01.431639,NaN,NaN,GradientBoosting,NaN,NaN,0.113596,8.0,7.0,5.0,188.0,COMPLETE


In [44]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,78
RandomForest,12
GradientBoosting,10


In [45]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.745996
RandomForest,0.768312
SVM,0.774746


In [46]:
# 1. Optimization History
plot_optimization_history(study).show()

In [47]:
# 3. Slice Plot
plot_slice(study).show()

In [48]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [53]:
#YET TO LEARN ABOUT THE XGBOOST AND THEN OPTUNA
      # FOR IT'S DETAILED HYPERPARAMETER TUNING

# import optuna
# import xgboost as xgb
# from sklearn.model_selection import train_test_split
# from sklearn.datasets import load_iris
# from sklearn.metrics import accuracy_score
# import numpy as np

# # Load the Iris dataset
# X, y = load_iris(return_X_y=True)

# # Split the dataset into training and test sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Define the objective function for XGBoost
# def objective(trial):
#     # Hyperparameter search space
#     param = {
#         'verbosity': 0,
#         'objective': 'multi:softprob',
#         'num_class': 3,
#         'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
#         'booster': 'gbtree',
#         'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
#         'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
#         'eta': trial.suggest_float('eta', 0.01, 0.3),
#         'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
#         'max_depth': trial.suggest_int('max_depth', 3, 9),
#         'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
#         'subsample': trial.suggest_float('subsample', 0.4, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
#         'n_estimators': 300,
#     }

#     # Create DMatrix for XGBoost
#     dtrain = xgb.DMatrix(X_train, label=y_train)
#     dtest = xgb.DMatrix(X_test, label=y_test)

#     # Define a pruning callback based on evaluation metrics
#     pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

#     # Train the model
#     bst = xgb.train(
#         param,
#         dtrain,
#         num_boost_round=300,
#         evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
#         early_stopping_rounds=30,
#         callbacks=[pruning_callback]
#     )

#     # Predict on the test set
#     preds = bst.predict(dtest)
#     best_preds = [int(np.argmax(line)) for line in preds]

#     # Return accuracy as the objective value
#     accuracy = accuracy_score(y_test, best_preds)
#     return accuracy

# # Create a study with pruning
# study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
# study.optimize(objective, n_trials=50)

# # Output the best trial
# print(f"Best trial: {study.best_trial.params}")
# print(f"Best accuracy: {study.best_value}")


In [54]:
# ! pip install optuna-integration[xgboost]

In [55]:
# from optuna.visualization import plot_intermediate_values

# # 1. Plot intermediate values during the trials
# plot_intermediate_values(study).show()